# Customer Churn Prediction – Feature Engineering

## Objective

The objective of this notebook is to transform the cleaned customer
churn dataset into a machine-learning-ready dataset.

## Feature Engineering Tasks

- Load the cleaned dataset
- Validate the target variable
- Separate identifiers from predictive features
- Convert the target variable into binary format
- Identify numerical and categorical features
- Create meaningful business-derived features
- Review feature distributions
- Prepare feature and target matrices

## Data Leakage Prevention

Feature engineering will be performed carefully to avoid using
information that would not be available at prediction time.

The target variable will never be used as an input feature.

## 1. Import Libraries

Pandas and NumPy are used for data manipulation and numerical
operations.

In [1]:
import pandas as pd
import numpy as np

## 2. Load Cleaned Dataset

The cleaned dataset produced during the data-cleaning stage is loaded
as the starting point for feature engineering.

In [3]:
processed_path = "../data/processed/telco_customer_churn_cleaned.csv"

df = pd.read_csv(processed_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (7043, 21)


In [4]:
df.columns.tolist()

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn']

## 3. Separate Target Variable

`Churn` is the target variable that the machine-learning model will
predict.

The target is separated from the input features to prevent the target
from accidentally becoming part of the feature matrix.

In [5]:
y = df["Churn"].copy()

X = df.drop(columns=["Churn"]).copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (7043, 20)
Target shape: (7043,)


## 4. Encode Target Variable

The categorical target variable is converted into binary numerical
values.

- `No` represents customers who did not churn and is encoded as `0`.
- `Yes` represents customers who churned and is encoded as `1`.

In [6]:
y = y.map({
    "No": 0,
    "Yes": 1
})

In [7]:
y.value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [8]:
y.isnull().sum()

np.int64(0)

## 5. Remove Identifier from Predictive Features

`customerID` is a unique identifier rather than a behavioral or
financial customer characteristic.

It is retained in the original dataset for traceability but excluded
from the machine-learning feature matrix.

In [9]:
X = X.drop(columns=["customerID"])

In [10]:
X.columns.tolist()

['gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges']

In [11]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 6. Create Average Monthly Spend

A derived feature called `AverageMonthlySpend` is created to represent
the customer's historical average monthly spending.

The calculation uses `TotalCharges` divided by `tenure`.

Customers with zero tenure cannot have a meaningful historical monthly
average, so division by zero is avoided.

In [12]:
df["AverageMonthlySpend"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    df["MonthlyCharges"]
)

In [13]:
df[
    [
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "AverageMonthlySpend"
    ]
].head(10)

,tenure,MonthlyCharges,TotalCharges,AverageMonthlySpend
0,1,29.85,29.85,29.850000
1,34,56.95,1889.50,55.573529
2,2,53.85,108.15,54.075000
3,45,42.30,1840.75,40.905556
4,2,70.70,151.65,75.825000
5,8,99.65,820.50,102.562500
6,22,89.10,1949.40,88.609091
7,10,29.75,301.90,30.190000
8,28,104.80,3046.05,108.787500
9,62,56.15,3487.95,56.257258


In [14]:
df["AverageMonthlySpend"].describe()

count    7043.000000
mean       64.762906
std        30.189796
min        13.775000
25%        35.935156
50%        70.337500
75%        90.174158
max       121.400000
Name: AverageMonthlySpend, dtype: float64

In [15]:
X["AverageMonthlySpend"] = df["AverageMonthlySpend"]

In [16]:
X.columns.tolist()

['gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'AverageMonthlySpend']

In [17]:
X.isnull().sum()

gender                 0
SeniorCitizen          0
Partner                0
Dependents             0
tenure                 0
PhoneService           0
MultipleLines          0
InternetService        0
OnlineSecurity         0
OnlineBackup           0
DeviceProtection       0
TechSupport            0
StreamingTV            0
StreamingMovies        0
Contract               0
PaperlessBilling       0
PaymentMethod          0
MonthlyCharges         0
TotalCharges           0
AverageMonthlySpend    0
dtype: int64

In [18]:
X.isnull().sum().sum()

np.int64(0)

In [19]:
np.isinf(
    X.select_dtypes(include=np.number)
).sum().sum()

np.int64(0)

In [20]:
modeling_df = X.copy()

modeling_df["Churn"] = y

In [21]:
modeling_path = "../data/processed/telco_customer_churn_features.csv"

modeling_df.to_csv(
    modeling_path,
    index=False
)

print("Feature-engineered dataset saved successfully.")
print("Path:", modeling_path)

Feature-engineered dataset saved successfully.
Path: ../data/processed/telco_customer_churn_features.csv
